# Milestone 3: base-model baselines (Qwen3-4B, Qwen3-1.7B) on a free Kaggle GPU

Runs the untrained base models on the 500-question held-out sample with vLLM,
through the same `xbrlbench run` code path as the frontier models
(`--api-base` pointed at a local vLLM server), each in thinking and
non-thinking mode.

**Setup (once):**
1. Kaggle → *Create* → *New notebook* → *File* → *Import notebook* → this file.
2. *Settings*: Accelerator **GPU T4 x2** (a single T4 or L4 also works, just slower), Internet **on**. Both need a phone-verified Kaggle account.
3. *Add input* → *Upload* a dataset containing `heldout_500.jsonl`
   (from `data/filings/v3/` on the machine that built the corpus).
4. *Run all*. About 1.5 hours. Then download `/kaggle/working/m3_base.zip` from the *Output* tab.

No API keys are needed and nothing is billed.

In [ ]:
BRANCH = "claude/project-thread-kxrvn1"   # branch with the M3 tooling (main once PR #11 merges)
CONFIGS = [  # (Hugging Face model, served name, --reasoning); fast non-thinking runs first
    ("Qwen/Qwen3-1.7B", "qwen3-1.7b-nothink", "off"),
    ("Qwen/Qwen3-1.7B", "qwen3-1.7b", "default"),
    ("Qwen/Qwen3-4B", "qwen3-4b-nothink", "off"),
    ("Qwen/Qwen3-4B", "qwen3-4b", "default"),
]
MAX_TOKENS = 8192      # thinking traces longer than this are cut off and graded unparseable
WORKERS = 64           # concurrent requests; vLLM batches them
TIMEOUT = 3600         # seconds per request: with 64 in flight, one long thinking trace can take many minutes
LIMIT = None           # e.g. 20 for a quick smoke run

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
pip install -q "vllm==0.31.0"
rm -rf /kaggle/working/XBRLBench
git clone -q https://github.com/allisonzhan/XBRLBench /kaggle/working/XBRLBench

In [ ]:
import glob
import subprocess

subprocess.run(["git", "-C", "/kaggle/working/XBRLBench", "checkout", "-q", BRANCH], check=True)
subprocess.run(["pip", "install", "-q", "-e", "/kaggle/working/XBRLBench"], check=True)
found = glob.glob("/kaggle/input/**/heldout_500.jsonl", recursive=True)
assert found, "Add a dataset containing heldout_500.jsonl (see the setup steps above)"
HELDOUT = found[0]
print("held-out file:", HELDOUT, sum(1 for _ in open(HELDOUT)), "questions")

In [ ]:
import os
import signal
import time
import urllib.request

# 1 or 2 GPUs both work; T4 x2 is fastest. (P100 is too old for vLLM.)
N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
assert N_GPUS >= 1, "No GPU: set Settings -> Accelerator to a GPU"


def start_server(hf_model, names):
    log = open(f"/kaggle/working/vllm_{names[0]}.log", "w")
    proc = subprocess.Popen(
        ["vllm", "serve", hf_model, "--served-model-name", *names,
         "--dtype", "half",                 # T4 has no bfloat16
         "--tensor-parallel-size", str(N_GPUS),
         "--max-model-len", str(MAX_TOKENS + 4096),
         "--gpu-memory-utilization", "0.90",
         "--port", "8000"],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2)
            return proc
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError(open(log.name).read()[-3000:]) from None
            time.sleep(5)
    raise RuntimeError("vLLM did not start in 15 minutes; see " + log.name)

def stop_server(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    proc.wait(timeout=120)
    time.sleep(10)

In [ ]:
OUT = "/kaggle/working/m3_base"


def run_streaming(cmd, **kw):
    """Run cmd and print its output line by line, so progress shows up in a
    committed run's Logs (a plain subprocess.run writes past the notebook)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


by_model = {}
for hf, name, reasoning in CONFIGS:
    by_model.setdefault(hf, []).append((name, reasoning))

for hf, runs in by_model.items():
    proc = start_server(hf, [n for n, _ in runs])
    try:
        for name, reasoning in runs:
            cmd = ["python", "-m", "xbrlbench", "run", "--in", HELDOUT,
                   "--api-base", "http://localhost:8000/v1", "--models", name,
                   "--reasoning", reasoning, "--max-tokens", str(MAX_TOKENS),
                   "--workers", str(WORKERS), "--timeout", str(TIMEOUT),
                   "--out-dir", f"{OUT}/{name}"]
            if LIMIT:
                cmd += ["--limit", str(LIMIT)]
            if os.path.exists(f"{OUT}/{name}/responses.jsonl"):
                cmd.append("--resume")
            t0 = time.time()
            run_streaming(cmd, cwd="/kaggle/working/XBRLBench")
            print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
            # Zip after every run, so a later failure still leaves results to download.
            subprocess.run(["bash", "-c", "cd /kaggle/working && rm -f m3_base.zip && zip -qr m3_base.zip m3_base"],
                           check=False)
    finally:
        stop_server(proc)

In [ ]:
# Combined report, then zip everything for download.
import pathlib

rows = []
for p in sorted(pathlib.Path(OUT).glob("*/responses.jsonl")):
    rows += p.read_text().splitlines(keepends=True)
pathlib.Path(f"{OUT}/responses.jsonl").write_text("".join(rows))
subprocess.run(["python", "-m", "xbrlbench", "report", "--in", f"{OUT}/responses.jsonl",
                "--questions", HELDOUT, "--out-prefix", f"{OUT}/report"],
               check=True, cwd="/kaggle/working/XBRLBench")
print(open(f"{OUT}/report.md").read())
subprocess.run(["bash", "-c", "cd /kaggle/working && rm -f m3_base.zip && zip -qr m3_base.zip m3_base vllm_*.log"], check=True)